# REF
https://scikit-learn.org/stable/auto_examples/applications/plot_time_series_lagged_features.html

https://stackoverflow.com/questions/67072953/how-to-add-lag-to-time-series-data

https://www.kaggle.com/c/bike-sharing-demand/data?select=train.csv

# IMPORT LIBRARY

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import glob
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.impute import SimpleImputer
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split, cross_val_score
from statistics import mean
import seaborn as sns
pd.options.mode.chained_assignment = None  # default='warn'

In [ ]:

import pandas as pd
import zipfile

# Đường dẫn đến file .zip trên Google Colab
zip_file_path = "D:\old data\VHL Project\Bio data\metadata-gga-txt-06112024.zip"

# Tên file .csv bên trong file .zip
csv_file_name = "metadata-gga-txt.csv"

# Mở file .zip và đọc file .csv bên trong
with zipfile.ZipFile(zip_file_path, 'r') as z:
    with z.open(csv_file_name) as f:
        all_data = pd.read_csv(f)

# Hiển thị các dòng đầu tiên của DataFrame
print(all_data.head())

In [ ]:
# sample_names = all_data['Sample_name'].unique()
# train_sample_names, test_sample_names = train_test_split(sample_names, test_size=0.4, random_state=42)

# train_data = all_data[all_data['Sample_name'].isin(train_sample_names)]
# test_data = all_data[all_data['Sample_name'].isin(test_sample_names)]

In [ ]:
# Tính số lượng hàng trong mỗi sample_name
sample_name_counts = all_data['Sample_name'].value_counts()

# Hiển thị số lượng hàng cho mỗi sample_name
print(sample_name_counts)

mean_count = sample_name_counts.mean()
print(mean_count)

# Plot phân phối số lượng hàng
plt.figure(figsize=(12, 6))
ax = sample_name_counts.plot(kind='hist', bins=30, edgecolor='black')
plt.xlabel('Số lượng hàng')
plt.ylabel('Số lượng sample_name')
plt.title('Phân phối số lượng hàng trong mỗi sample_name')

# Thêm chú thích lên mỗi cột
bin_edges = ax.patches
for bin in bin_edges:
    height = bin.get_height()
    if height > 0:
        ax.text(bin.get_x() + bin.get_width() / 2, height, f'{int(height)}', ha='center', va='bottom')

plt.grid(True)
plt.show()

In [ ]:
# Tính số lượng hàng trong mỗi sample_name
sample_name_counts = all_data['Sample_name'].value_counts()

# Tính giá trị trung bình của số lượng hàng và làm tròn
mean_count = round(sample_name_counts.mean())
print("Giá trị trung bình của số lượng hàng trong mỗi sample_name:", mean_count)

# Tính giá trị trung bình của cột "DO"
mean_do = all_data['DO'].mean()
print("Giá trị trung bình của toàn bộ giá trị ở cột 'DO':", mean_do)

# Hàm để thêm các hàng vào sample_name
def add_rows_to_sample_name(data, sample_name, target_count, mean_do):
    current_count = len(data)
    num_rows_to_add = target_count - current_count
    if num_rows_to_add > 0:
        last_time = int(data['Time'].max())
        new_times = range(last_time + 1, last_time + 1 + num_rows_to_add)
        new_data = pd.DataFrame({'Time': new_times, 'DO': mean_do, 'Sample_name': sample_name})
        data = pd.concat([data, new_data], ignore_index=True)
    return data

# Hàm để bỏ bớt các hàng từ giây 6574 trở đi
def remove_rows_from_sample_name(data, target_count):
    if len(data) > target_count:
        data = data.iloc[:target_count]
    return data

# Tạo DataFrame mới với các hàng đã điều chỉnh
new_all_data = pd.DataFrame()

for sample_name in sample_name_counts.index:
    sample_data = all_data[all_data['Sample_name'] == sample_name]
    if len(sample_data) < mean_count:
        sample_data = add_rows_to_sample_name(sample_data, sample_name, mean_count, mean_do)
    elif len(sample_data) > mean_count:
        sample_data = remove_rows_from_sample_name(sample_data, mean_count)
    new_all_data = pd.concat([new_all_data, sample_data], ignore_index=True)

# Plot phân phối số lượng hàng trong new_all_data
new_sample_name_counts = new_all_data['Sample_name'].value_counts()

# Plot phân phối số lượng hàng
plt.figure(figsize=(12, 6))
ax = new_sample_name_counts.plot(kind='hist', bins=30, edgecolor='black')
plt.xlabel('Số lượng hàng')
plt.ylabel('Số lượng sample_name')
plt.title('Phân phối số lượng hàng trong mỗi sample_name')

# Thêm chú thích lên mỗi cột
bin_edges = ax.patches
for bin in bin_edges:
    height = bin.get_height()
    if height > 0:
        ax.text(bin.get_x() + bin.get_width() / 2, height, f'{int(height)}', ha='center', va='bottom')

plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(new_all_data['DO'], kde=True, bins=50, color = "blue")
plt.xlabel('Giá trị DO')
plt.ylabel('Tần suất')
plt.title('Phân phối giá trị cột DO trong tất cả các sample_name')
plt.grid(True)
plt.show()

In [ ]:
# Tìm các sample_name có ít nhất một giá trị DO nhỏ hơn 200
sample_names_with_do_less_than_200 = new_all_data[new_all_data['DO'] < 200]['Sample_name'].unique()
print(sample_names_with_do_less_than_200)
# Loại bỏ các sample_name này khỏi all_data
filtered_all_data = new_all_data[~new_all_data['Sample_name'].isin(sample_names_with_do_less_than_200)]

plt.figure(figsize=(12, 6))
sns.histplot(filtered_all_data['DO'], kde=True, bins=50, color = "blue")
plt.xlabel('Giá trị DO')
plt.ylabel('Tần suất')
plt.title('Phân phối giá trị cột DO trong tất cả các sample_name')
plt.grid(True)
plt.show()

In [ ]:
sample_names = filtered_all_data['Sample_name'].unique()
train_sample_names, test_sample_names = train_test_split(sample_names, test_size=0.4, random_state=42)

train_data = all_data[all_data['Sample_name'].isin(train_sample_names)]
test_data = all_data[all_data['Sample_name'].isin(test_sample_names)]
train_data.reset_index(drop=True, inplace=True)
test_data.reset_index(drop=True, inplace=True)

In [ ]:
train_data

In [ ]:
train_features = []
train_labels = []
test_features = []
test_labels = []

for sample_name, group in train_data.groupby('Sample_name'):
    do_mean = group['DO'].mean()
    do_std = group['DO'].std()
    do_max = group['DO'].max()
    do_min = group['DO'].min()

    label = group["DO"].iloc[0]

    train_features.append([do_mean, do_std, do_max, do_min])
    train_labels.append(label)

for sample_name, group in test_data.groupby('Sample_name'):
    do_mean = group['DO'].mean()
    do_std = group['DO'].std()
    do_max = group['DO'].max()
    do_min = group['DO'].min()

    label = group["DO"].iloc[0]

    test_features.append([do_mean, do_std, do_max, do_min])
    test_labels.append(label)

X_train_stats = pd.DataFrame(train_features, columns=['do_mean', 'do_std', 'do_max', 'do_min'])
y_train_stats = pd.Series(train_labels)
X_test_stats = pd.DataFrame(test_features, columns=['do_mean', 'do_std', 'do_max', 'do_min'])
y_test_stats = pd.Series(test_labels)

## Create LAG Feature

In [ ]:

window_size = 8
for lag in range(1, window_size + 1):
    # all_data[f'DO_lag_{lag}'] = all_data['DO'].shift(lag)
    train_data.loc[:, f'DO_lag_{lag}'] = train_data['DO'].shift(lag)
    test_data.loc[:, f'DO_lag_{lag}'] = test_data['DO'].shift(lag)

# Drop các hàng có giá trị NaN và reset lại chỉ số
train_data = train_data.dropna().reset_index(drop=True)
test_data = test_data.dropna().reset_index(drop=True)
print(train_data)

In [ ]:
X_train_lag = train_data[[f'DO_lag_{i}' for i in range(1, window_size + 1)]]
y_train_lag = train_data['DO']
X_test_lag = test_data[[f'DO_lag_{i}' for i in range(1, window_size + 1)]]
y_test_lag = test_data['DO']

In [ ]:
# X_train = pd.concat([X_train_stats.reset_index(drop=True), X_train_lag.reset_index(drop=True)], axis=1)
# X_test = pd.concat([X_test_stats.reset_index(drop=True), X_test_lag.reset_index(drop=True)], axis=1)

In [ ]:
X_train_lag

In [ ]:
y_train_lag

In [ ]:
# imputer = SimpleImputer(strategy='mean')
# X_train = imputer.fit_transform(X_train)
# X_test = imputer.transform(X_test)

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train_lag, y_train_lag)

y_pred = lr_model.predict(X_test_lag)

In [ ]:
X_test_lag

In [ ]:
mse = mean_squared_error(y_test_lag, y_pred)
mae = mean_absolute_error(y_test_lag, y_pred)
print("Mean Squared Error (MSE):", mse)
print("Mean Absolute Error (MAE):", mae)

In [ ]:
plt.figure(figsize=(20,5))
plt.plot(y_test_lag[0:100], label='Actual DO')
plt.plot(y_pred[0:100], label='Predicted DO')
plt.xlabel('Sample Index')
plt.ylabel('DO')
plt.title('Actual vs Predicted DO')
plt.legend()
plt.show()

## Test with 1 sample

In [ ]:
test_1_sample_data = all_data[all_data['Sample_name']== 'U100-H3-VS2-7.5-2.5-03072024-Q=50.66mL-phút-5.txt']
test_1_sample_data.reset_index(drop=True, inplace=True)
test_1_sample_data.drop(['Time', 'Sample_name'], axis = 1, inplace = True)
test_1_sample_data

In [ ]:
import numpy as np
import pandas as pd

# Giả sử lr_model là model đã được train trước đó
# Xác định chỉ số bắt đầu và kết thúc
start_index = int(len(test_1_sample_data) * 0.6)
end_index = len(test_1_sample_data)
print(start_index)

y_test_1_sample = test_1_sample_data.iloc[start_index:]
print(len(y_test_1_sample))

# Cắt đi phần từ 3146 đến 5244 của test_1_sample_data
test_1_sample_data = test_1_sample_data.iloc[:start_index].copy()

window_size = 8
for lag in range(1, window_size + 1):
    test_1_sample_data.loc[:, f'DO_lag_{lag}'] = test_1_sample_data['DO'].shift(lag)

test_1_sample_data = test_1_sample_data.dropna().reset_index(drop=True)

# Danh sách để lưu trữ các giá trị dự đoán
predicted_values = []

# Thực hiện vòng lặp
for i in range(start_index, end_index):
    # Dự đoán giá trị cho toàn bộ DataFrame
    X_test_df = test_1_sample_data[[f'DO_lag_{j}' for j in range(1, window_size + 1)]]
    y_pred_df = lr_model.predict(X_test_df)
    
    # Lưu trữ các giá trị dự đoán vào danh sách
    predicted_values.extend(y_pred_df)
    
    # Lấy hàng cuối cùng từ test_1_sample_data
    last_row = test_1_sample_data.iloc[-1]
    
    # Chuẩn bị dữ liệu đầu vào cho model để dự đoán giá trị mới
    X_test_1_row = last_row[[f'DO_lag_{j}' for j in range(1, window_size + 1)]].to_frame().T
    
    # Dự đoán giá trị mới
    y_pred_1_row = lr_model.predict(X_test_1_row)[0]
    
    # Tạo hàng mới với giá trị dự đoán và cập nhật các giá trị lag
    new_row = np.roll(last_row[[f'DO_lag_{j}' for j in range(1, window_size + 1)]].values, shift=1)
    new_row[0] = y_pred_1_row
    
    # Chuyển hàng mới thành DataFrame và nối vào test_1_sample_data
    new_row_df = pd.DataFrame([np.append(new_row, y_pred_1_row)], columns=[f'DO_lag_{j}' for j in range(1, window_size + 1)] + ['DO'])
    test_1_sample_data = pd.concat([test_1_sample_data, new_row_df], ignore_index=True)

# Kiểm tra kết quả cuối cùng
# print(predicted_values)
print(len(predicted_values))


In [ ]:
# # Giả sử lr_model là model đã được train trước đó
# # Xác định chỉ số bắt đầu và kết thúc
# start_index = int(len(test_1_sample_data) * 0.6)
# end_index = len(test_1_sample_data)
# print(start_index)

# y_test_1_sample = test_1_sample_data.iloc[start_index:]
# print(len(y_test_1_sample))

# # Cắt đi phần từ 3146 đến 5244 của test_1_sample_data
# test_1_sample_data = test_1_sample_data.iloc[:start_index].copy()

# window_size = 8
# for lag in range(1, window_size + 1):
#     test_1_sample_data.loc[:, f'DO_lag_{lag}'] = test_1_sample_data['DO'].shift(lag)

# test_1_sample_data = test_1_sample_data.dropna().reset_index(drop=True)

# # Danh sách để lưu trữ các giá trị dự đoán
# predicted_values = []

# # Thực hiện vòng lặp
# for i in range(start_index, end_index):
#     # Lấy hàng cuối cùng từ test_1_sample_data tại mỗi bước lặp
#     last_row = test_1_sample_data.iloc[-1]

#     # Chuẩn bị dữ liệu đầu vào cho model
#     X_test_1_row = last_row[[f'DO_lag_{j}' for j in range(1, window_size + 1)]].to_frame().T

#     # Dự đoán giá trị mới
#     y_pred_1_row = lr_model.predict(X_test_1_row)[0]

#     # Lưu trữ giá trị dự đoán vào danh sách
#     predicted_values.append(y_pred_1_row)

#     # Tạo hàng mới với giá trị dự đoán và cập nhật các giá trị lag
#     new_row = np.roll(last_row[[f'DO_lag_{j}' for j in range(1, window_size + 1)]].values, shift=1)
#     new_row[0] = y_pred_1_row

#     # Chuyển hàng mới thành DataFrame và nối vào test_1_sample_data
#     new_row_df = pd.DataFrame([np.append(new_row, y_pred_1_row)], columns=[f'DO_lag_{j}' for j in range(1, window_size + 1)] + ['DO'])
#     test_1_sample_data = pd.concat([test_1_sample_data, new_row_df], ignore_index=True)

# # Kiểm tra kết quả cuối cùng
# print(predicted_values)
# print(len(predicted_values))


In [ ]:
print(len(predicted_values))

In [ ]:
# x_size = int(len(test_1_sample_data) * 0.6)
# X_test_1_sample = test_1_sample_data[[f'DO_lag_{i}' for i in range(1, window_size + 1)]].iloc[:x_size]
# y_test_1_sample = test_1_sample_data['DO']

In [ ]:
# x_size = int(len(test_1_sample_data) * 0.6)
# x_test_1_sample_data = test_1_sample_data.iloc[:x_size]
# y_test_1_sample_data = test_1_sample_data.iloc[x_size:]

# # Kiểm tra kết quả
# print("Predict size:", len(x_test_1_sample_data))
# print("Test size:", len(y_test_1_sample_data))

# y_test_1_sample_data

In [ ]:
# y_pred_1_sample = lr_model.predict(X_test_1_sample)
mse = mean_squared_error(y_test_1_sample, predicted_values)
mae = mean_absolute_error(y_test_1_sample, predicted_values)
print("Mean Squared Error (MSE):", mse)
print("Mean Absolute Error (MAE):", mae)

plt.figure(figsize=(20,5))
plt.plot(y_test_1_sample, label='Actual DO')
plt.plot(predicted_values, label='Predicted DO')
plt.xlabel('Sample Index')
plt.ylabel('DO')
plt.title('Actual vs Predicted DO')
plt.legend()
plt.show()

# Not Worked

In [ ]:
# Trích xuất danh sách các sample_name
sample_names = all_data['Sample_name'].unique()

# Chia danh sách sample_names thành 60% train và 40% test
train_sample_names, test_sample_names = train_test_split(sample_names, test_size=0.4, random_state=42)

# Tạo tập train và test dựa trên sample_names đã chia
train_data = all_data[all_data['Sample_name'].isin(train_sample_names)]
test_data = all_data[all_data['Sample_name'].isin(test_sample_names)]

# Khởi tạo mô hình Linear Regression
lr_model = LinearRegression()
window_size = 12
imputer = SimpleImputer(strategy='mean')
# Huấn luyện mô hình trên từng file .txt trong tập train
for sample_name in train_sample_names:
    train_subset = train_data[train_data['Sample_name'] == sample_name].copy()

    # Tạo cột lag (độ trễ)
      # Kích thước cửa sổ
    for lag in range(1, window_size + 1):
        train_subset.loc[:, f'DO_lag_{lag}'] = train_subset['DO'].shift(lag)

    train_subset = train_subset.dropna()  # Loại bỏ các hàng có giá trị NaN

    # Chuẩn bị các đặc trưng và mục tiêu
    X_train = train_subset[[f'DO_lag_{i}' for i in range(1, window_size + 1)]]
    y_train = train_subset['DO']

    # Sử dụng SimpleImputer để điền giá trị NaN bằng giá trị trung bình

    X_train = imputer.fit_transform(X_train)

    # Train mô hình
    lr_model.fit(X_train, y_train)

# Dự đoán cho tập test
mse_scores = []
mae_scores = []

for sample_name in test_sample_names:
    test_subset = test_data[test_data['Sample_name'] == sample_name].copy()
    split_index = int(len(test_subset) * 0.6)

    # Dữ liệu trước đó
    past_data = test_subset[:split_index]

    # Tạo các cột lag cho dữ liệu trước đó
    for lag in range(1, window_size + 1):
        past_data.loc[:, f'DO_lag_{lag}'] = past_data['DO'].shift(lag)

    past_data = past_data.dropna()  # Loại bỏ các hàng có giá trị NaN

    X_past = past_data[[f'DO_lag_{i}' for i in range(1, window_size + 1)]]
    X_past = imputer.transform(X_past)  # Áp dụng imputer

    # Dự đoán giá trị tiếp theo
    num_predictions = len(test_subset) - split_index
    predictions = []
    last_known_values = X_past[-1].reshape(1, -1)

    for _ in range(num_predictions):
        next_pred = lr_model.predict(last_known_values)[0]
        predictions.append(next_pred)

        # Cập nhật last_known_values với giá trị dự đoán mới
        last_known_values = np.roll(last_known_values, -1)
        last_known_values[0, -1] = next_pred

    # Thực tế
    actual_values = test_subset['DO'][split_index:split_index + num_predictions]

    # Tính toán MSE và MAE
    mse = mean_squared_error(actual_values, predictions)
    mae = mean_absolute_error(actual_values, predictions)

    mse_scores.append(mse)
    mae_scores.append(mae)

# Trung bình kết quả predict trên 63 file test
mean_mse = np.mean(mse_scores)
mean_mae = np.mean(mae_scores)

print("Mean Squared Error (MSE):", mean_mse)
print("Mean Absolute Error (MAE):", mean_mae)


In [ ]:
train_subset[0:10]

# OLD

## READ FILE

In [ ]:
# Initialize an empty DataFrame to store data from all files
all_data = pd.DataFrame()

def process_file_without_bom(file):
    # Initialize empty lists for Time and DO columns
    Time_list = []
    DO_list = []

    # Open file to read
    with open(file, "r") as f:
        # Read each line in the file
        for line in f:
            # Split the line into values based on space
            parts = line.split()
            if len(parts) < 2:
                continue
            # Add values to the corresponding lists
            Time_list.append(float(parts[0].replace("\x00","")))
            DO_list.append(float(parts[1].replace("\x00","")))

    # Create a dictionary from the lists
    temp_dict = {"Time": Time_list, "DO": DO_list}
    # Convert the dictionary to a DataFrame
    temp_data = pd.DataFrame(temp_dict)
    return temp_data

# Iterate over all txt files in the directory and subdirectories
directory = "BOD2024-Nhung/GGA/File txt"
offset = 0

for file in glob.glob(os.path.join(directory, "**/*.txt"), recursive=True):
    print(f"Reading file: {file}")
    try:
        # Attempt to read file with UTF-16 encoding
        temp_data = pd.read_csv(file, sep="\t", header=None, names=["Time", "DO"], encoding='utf-16')
    except UnicodeError as e:
        if "UTF-16 stream does not start with BOM" in str(e):
            # Process the file without BOM if UTF-16 fails
            temp_data = process_file_without_bom(file)
        else:
            print(f"Error reading {file} with UTF-16: {e}")
            continue

    # Add the file name as a new column 'Sample_name'
    temp_data["Sample_name"] = os.path.basename(file).strip()

    # Add a new column 'TrueTime' to store the actual time without offset
    temp_data['TrueTime'] = temp_data['Time'] + offset # 1 - 7000, 7001

    # Adjust the Time column to avoid overlaps
    temp_data['Time'] = temp_data['Time'] + offset
    offset += temp_data['Time'].max()  # Update offset

    # Append the data to the all_data DataFrame
    all_data = pd.concat([all_data, temp_data], ignore_index=True)

# Reset the index of the final DataFrame
all_data.reset_index(drop=True, inplace=True)

## PLOT

In [ ]:
# Plot
plt.figure(figsize=(20,5))
for sample_name in all_data['Sample_name'].unique():
    subset = all_data[all_data['Sample_name'] == sample_name]
    plt.plot(subset['Time'], subset['DO'], label=sample_name)

plt.xlabel('Time')
plt.ylabel('DO')
plt.title('Combined Time Series Data')
plt.legend()
plt.show()

## Prepare Data for Train Test

In [ ]:
window_size = 12  # Window Size
for lag in range(1, window_size + 1):
    all_data[f'DO_lag_{lag}'] = all_data['DO'].shift(lag)

all_data = all_data.dropna()  # Remove NaN

# Split to train set, test set
train_len = round(len(all_data) * 0.6)
train = all_data[:train_len]
test = all_data[train_len:]

# Prepare features and target
X_train = train[[f'DO_lag_{i}' for i in range(1, window_size + 1)]]
y_train = train['DO']
X_test = test[[f'DO_lag_{i}' for i in range(1, window_size + 1)]]
y_test = test['DO']

print("Training data shape:", X_train.shape)
print("Test data shape:", X_test.shape)

## TEST

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.impute import SimpleImputer

# Đọc dữ liệu từ file .csv
file_path = "/content/metadata-gga-txt.csv"  # Thay đổi đường dẫn đến file .csv của bạn
all_data = pd.read_csv(file_path)

# Plot toàn bộ dữ liệu
# plt.figure(figsize=(20,5))
# for sample_name in all_data['Sample_name'].unique():
#     subset = all_data[all_data['Sample_name'] == sample_name]
#     plt.plot(subset['Time'], subset['DO'], label=sample_name)

# plt.xlabel('Time')
# plt.ylabel('DO')
# plt.title('Combined Time Series Data')
# plt.legend()
# plt.show()

# Trích xuất các đặc trưng thống kê từ từng mẫu
features = []
labels = []

for sample_name, group in all_data.groupby('Sample_name'):
    time_mean = group['Time'].mean()
    time_std = group['Time'].std()
    time_max = group['Time'].max()
    time_min = group['Time'].min()

    do_mean = group['DO'].mean()
    do_std = group['DO'].std()
    do_max = group['DO'].max()
    do_min = group['DO'].min()

    features.append([time_mean, time_std, time_max, time_min, do_mean, do_std, do_max, do_min])
    labels.append(do_mean)  # Sử dụng trung bình của DO làm label cho thống kê

# Chuyển đổi thành DataFrame và Series
X_stats = pd.DataFrame(features, columns=['time_mean', 'time_std', 'time_max', 'time_min', 'do_mean', 'do_std', 'do_max', 'do_min'])
y_stats = pd.Series(labels)

# Tạo cột lag (độ trễ)
window_size = 12  # Kích thước cửa sổ
for lag in range(1, window_size + 1):
    all_data[f'DO_lag_{lag}'] = all_data['DO'].shift(lag)

all_data = all_data.dropna()  # Loại bỏ các hàng có giá trị NaN

# Tách dữ liệu thành tập train và test dựa trên các đặc trưng lag và các giá trị của cột "DO"
train_len = round(len(all_data) * 0.6)
train = all_data[:train_len]
test = all_data[train_len:]

X_lag = all_data[[f'DO_lag_{i}' for i in range(1, window_size + 1)]]
y_lag = all_data['DO']

# Chuẩn bị các tập train và test
X_train_stats, X_test_stats, y_train_stats, y_test_stats = train_test_split(X_stats, y_stats, test_size=0.4, random_state=42)
X_train_lag, X_test_lag, y_train_lag, y_test_lag = train_test_split(X_lag, y_lag, test_size=0.4, random_state=42)

# Kết hợp các đặc trưng thống kê và lag thành các tập train và test
X_train = pd.concat([X_train_stats.reset_index(drop=True), X_train_lag.reset_index(drop=True)], axis=1)
X_test = pd.concat([X_test_stats.reset_index(drop=True), X_test_lag.reset_index(drop=True)], axis=1)
y_train = y_train_lag.reset_index(drop=True)  # Sử dụng y_train_lag vì nó chứa giá trị thực DO
y_test = y_test_lag.reset_index(drop=True)  # Sử dụng y_test_lag vì nó chứa giá trị thực DO

# Sử dụng SimpleImputer để điền giá trị NaN bằng giá trị trung bình
imputer = SimpleImputer(strategy='mean')
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

# Huấn luyện mô hình Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Dự báo trên tập test
y_pred = lr_model.predict(X_test)

# Đánh giá mô hình
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print("Mean Squared Error (MSE):", mse)
print("Mean Absolute Error (MAE):", mae)

# Hiển thị dự báo so với giá trị thực
plt.figure(figsize=(20,5))
plt.plot(y_test.values, label='Actual DO')
plt.plot(y_pred, label='Predicted DO')
plt.xlabel('Sample Index')
plt.ylabel('DO')
plt.title('Actual vs Predicted DO')
plt.legend()
plt.show()
